In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
# 1) Define the ranges
num_points = 10000
F_s_vals = np.linspace(-3, 3, num_points)

In [ ]:
from matplotlib.ticker import MultipleLocator
import matplotlib.pyplot as plt

def plot_regularisation(in_vals, R_vals, ref_vals=None,
                        x_label=r'$F_s$', title="", ylim=None,
                        labels=None, colors=None):
    """
    Plot multiple regularisation curves R versus ΔF_s in one figure.

    Parameters
    ----------
    colors : list, optional
        List of matplotlib colors for each curve.
    """

    plt.figure(figsize=(6, 4))

    if ref_vals is not None:
        for idx, r_vals in enumerate(R_vals):
            label = labels[idx] if labels is not None else None
            color = colors[idx] if colors is not None else None

            plt.plot(
                in_vals,
                r_vals,
                label=label,
                linewidth=5,
                color=color
            )

        if labels is not None:
            plt.legend(frameon=True, fontsize=12)

    else:
        color = colors[0] if colors is not None else None
        plt.plot(in_vals, R_vals, label='R', linewidth=5, color=color)

    plt.xlabel(x_label, fontsize=18)
    plt.ylabel(r'$R^{\mathrm{abs}}$', fontsize=18)
    plt.title(title)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.gca().xaxis.set_major_locator(MultipleLocator(1))

    if ylim is not None:
        plt.ylim(ylim)

    plt.grid(True)
    plt.show()

REG STREAMLINE + COEFF

In [ ]:
def reg_coeff_streamline(F_s):

    reg = F_s**2

    return reg

# 3. Compute R for each ΔF_s in that range, using the fixed ΔF_mean
R_vals = reg_coeff_streamline(F_s_vals)
plot_regularisation(F_s_vals, R_vals, x_label=r'$F_s$', ylim=(0, 25))

REG STREAMLINE + FACTOR

In [ ]:
import numpy as np

def reg_factor_streamline(F_s):
    """
    Vectorized implementation of the factor function regularisation 
    on a streamline basis:

        R(F_s) = (exp(F_s) - 1)^2

    where F_s is the streamline coefficient and exp(F_s) is its weight.
    The penalty increases quadratically as the factor deviates from one.
    """
    eF_s = np.exp(F_s)

    # Compute factor penalty
    R = (eF_s - 1.0)**2

    return R

R_vals = reg_factor_streamline(F_s_vals)
plot_regularisation(F_s_vals, R_vals, x_label=r'$F_s$',ylim=(0, 25))

STREAMLINE + GAMMA

In [ ]:
import numpy as np

def reg_gamma_streamline(F_s):
    """
    Vectorized implementation of the gamma function regularisation 
    on a streamline basis:

        R(F_s) = F_s^2                 , if F_s <= 0
                 (exp(F_s) - 1)^2      , if F_s > 0

    where F_s is the streamline coefficient and exp(F_s) its 
    cross-sectional weight. This combines the coefficient and factor 
    penalties, imposing heavier costs for F_s > 0.
    """
    F_s = np.array(F_s, dtype=float)

    # Branch 1: coefficient penalty (F_s <= 0)
    branch1 = F_s**2

    # Branch 2: factor penalty (F_s > 0)
    branch2 = (np.exp(F_s) - 1.0)**2

    # Apply condition
    R = np.where(F_s <= 0, branch1, branch2)

    return R

R_vals = reg_gamma_streamline(F_s_vals)
plot_regularisation(F_s_vals, R_vals, x_label = r'$F_s$',ylim=(0,25))


FIXEL/GROUP + COEFF

In [ ]:
import numpy as np

def reg_coeff_fixel_group(F_s, F_mean):
    """
    Vectorized implementation of the coefficient function 
    on a fixel/group basis:

        R(F_s, F_mean) = (F_s - F_mean)^2

    where:
      - F_s is the streamline coefficient (array-like)
      - F_mean is the mean coefficient of all streamlines 
        assigned to the same fixel or group (scalar or array)

    This imposes a quadratic penalty as a streamline's coefficient 
    deviates from the local mean.
    """

    # Quadratic penalty relative to local mean
    R = (F_s - F_mean) ** 2

    return R

ref_vals = [-2, 0, 2]
labels = [fr"${{F}}_{{\mathrm{{ref}}}}={ref_val:g}$" for ref_val in ref_vals]

R_vals = [reg_coeff_fixel_group(F_s_vals, F_mean=ref_vals[0]),reg_coeff_fixel_group(F_s_vals, F_mean=ref_vals[1]),reg_coeff_fixel_group(F_s_vals, F_mean=ref_vals[2])]
plot_regularisation(F_s_vals, R_vals, ref_vals=ref_vals, x_label=r'$F_s$',ylim=(0, 25),labels=labels)


FIXEL/GROUP + FACTOR

In [ ]:
import numpy as np

def reg_factor_fixel_group(F_s, eF_mean):
    """
    Vectorized implementation of the factor function on a fixel/group basis:

        R(F_s, F_mean) = (exp(F_s) - exp(F_mean))^2

    where:
      - F_s is the streamline coefficient (array-like)
      - F_mean is the mean coefficient of streamlines assigned 
        to the same fixel or group (scalar or array)

    This penalises quadratic deviations of streamline weights (exp(F_s)) 
    from the mean streamline weight (exp(F_mean)), with larger penalties 
    for deviations due to the exponential transform.
    """

    eF_s = np.exp(F_s)

    # Quadratic penalty in factor domain
    R = (eF_s - eF_mean) ** 2

    return R

ref_vals = [0.01, 1, 2]
labels = [fr"${{e^{{F_{{\mathrm{{ref}}}}}}}}={ref_val:g}$" for ref_val in ref_vals]

R_vals = [reg_factor_fixel_group(F_s_vals, eF_mean=ref_vals[0]),reg_factor_fixel_group(F_s_vals, eF_mean=ref_vals[1]),reg_factor_fixel_group(F_s_vals, eF_mean=ref_vals[2])]
plot_regularisation(F_s_vals, R_vals, ref_vals=ref_vals, x_label=r'$F_s$', ylim=(0,25), labels=labels)

FIXEL/GROUP + GAMMA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def reg_gamma_fixel_group(F_s, F_mean):
    """
    Vectorized implementation of the gamma function on a fixel/group basis:

        R(F_s, F_mean) = (F_s - F_mean)^2                , if F_s <= F_mean
                         (exp(F_s) - exp(F_mean))^2      , if F_s > F_mean

    where:
      - F_s is the streamline coefficient (array-like)
      - exp(F_s) is the streamline weight
      - F_mean is the mean streamline coefficient of streamlines in the same fixel/group
      - exp(F_mean) is the mean streamline weight

    This switches between a coefficient penalty (for coefficients below or equal to
    the mean) and a factor penalty (for coefficients above the mean), thereby
    imposing heavier penalties on large positive deviations.
    """
    F_s = np.array(F_s, dtype=float)
    F_mean = np.array(F_mean, dtype=float)

    # Coefficient penalty (Fs <= F_mean)
    branch1 = (F_s - F_mean) ** 2

    # Factor penalty (Fs > F_mean)
    branch2 = (np.exp(F_s) - np.exp(F_mean)) ** 2

    # Piecewise combination
    R = np.where(F_s <= F_mean, branch1, branch2)

    return R

ref_vals = [-2, 0, 2]
labels = [fr"${{F}}_{{\mathrm{{ref}}}}={ref_val:g}$" for ref_val in ref_vals]

R_vals = [reg_gamma_fixel_group(F_s_vals, F_mean=ref_vals[0]),reg_gamma_fixel_group(F_s_vals, F_mean=ref_vals[1]),reg_gamma_fixel_group(F_s_vals, F_mean=ref_vals[2])]
plot_regularisation(F_s_vals, R_vals, ref_vals=ref_vals, x_label=r'$F_s$', ylim=(0,25), labels=labels)